# J.A.R.V.I.S. Unsloth Fine-Tuning (Colab)

This notebook uses [Unsloth](https://github.com/unslothai/unsloth) to perform Supervised Fine-Tuning (SFT) on an Orion Agent model. 

**Hardware Required**: T4 GPU (Free tier is perfectly fine!)

In [10]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0cxmi8dt/unsloth_22abddb8d890429baa3e8268fb66cbbd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0cxmi8dt/unsloth_22abddb8d890429baa3e8268fb66cbbd
  Resolved https://github.com/unslothai/unsloth.git to commit 31ac558a73fd76bc200a4a22f10ceae5a73b9ae4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # Use 4bit quantization to save memory

# We recommend Qwen2.5-7B or Llama-3-8B for agentic tasks
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [12]:
# Setup LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Load your dataset
Make sure you upload `orion_synthetic_sft.jsonl` (or your extracted `orion_sft_dataset.jsonl`) to the Colab files pane.

In [13]:
import json
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Create the dataset file directly on the Colab server
synthetic_data = [
    {"messages": [{"role": "user", "content": "Orion, open Chrome."}, {"role": "assistant", "content": "I will open Google Chrome for you now.", "tool_calls": [{"id": "call_1", "type": "function", "function": {"name": "shell_exec", "arguments": "{\"command\": \"start chrome\"}"}}]}]},
    {"messages": [{"role": "user", "content": "What is the Sensex value?"}, {"role": "assistant", "content": "Let me check the web.", "tool_calls": [{"id": "call_2", "type": "function", "function": {"name": "web_search", "arguments": "{\"query\": \"current Sensex value BSE India\"}"}}]}]},
    {"messages": [{"role": "user", "content": "Open my hello orion note in Obsidian."}, {"role": "assistant", "content": "Searching your Obsidian vault.", "tool_calls": [{"id": "call_3", "type": "function", "function": {"name": "obsidian_search_notes", "arguments": "{\"query\": \"hello orion\"}"}}]}]}
]

with open('orion_synthetic_sft.jsonl', 'w') as f:
    for item in synthetic_data:
        f.write(json.dumps(item) + '\n')

# 2. Now load it!
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="orion_synthetic_sft.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Change to None and use num_train_epochs = 3 for full training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,7.968995
2,7.968995
3,7.968995
4,7.681737
5,6.595456
6,5.560833
7,5.559810
8,4.451885
9,3.494763
10,2.877244


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [15]:
# Export the LoRA adapters to load them back into Orion locally!
model.save_pretrained("orion_lora_model") # Local saving
tokenizer.save_pretrained("orion_lora_model")

# Zip the folder so you can download it from the Colab sidebar
!zip -r orion_lora_model.zip orion_lora_model/

Unsloth: Restored added_tokens_decoder metadata in orion_lora_model/tokenizer_config.json.


  adding: orion_lora_model/ (stored 0%)
  adding: orion_lora_model/tokenizer_config.json (deflated 90%)
  adding: orion_lora_model/adapter_model.safetensors (deflated 8%)
  adding: orion_lora_model/adapter_config.json (deflated 59%)
  adding: orion_lora_model/README.md (deflated 65%)
  adding: orion_lora_model/chat_template.jinja (deflated 59%)
  adding: orion_lora_model/tokenizer.json (deflated 81%)
